In [1]:
import duckdb

from dotenv import dotenv_values

config = dotenv_values(".env")

In [2]:
datum = '2026-03-01'

In [3]:
duck = duckdb.connect()

In [4]:
duck.sql(f"""INSTALL postgres;
                            LOAD postgres;
                            ATTACH 'dbname=zvbn_postgis user={config['POSTGRES_USER']} 
                            host=127.0.0.1 password={config['POSTGRES_PW']}' AS db_dm (TYPE POSTGRES, READ_ONLY);
                            """
                        )

In [5]:
duck.sql("""ATTACH '/home/zvbn/python/ivu/db/ivu_rt2.db' AS ivu;""")

In [6]:
duck.sql("describe ivu.rt").df()

,column_name,column_type,null,key,default,extra
0,datum,TIMESTAMP_NS,YES,None,None,None
1,linie,VARCHAR,YES,None,None,None
2,fahrzeug,BIGINT,YES,None,None,None
3,fahrt,DOUBLE,YES,None,None,None
4,kurs,DOUBLE,YES,None,None,None
5,nr,BIGINT,YES,None,None,None
6,hpkt,BIGINT,YES,None,None,None
7,sollabfahrt,VARCHAR,YES,None,None,None
8,sollab,VARCHAR,YES,None,None,None
9,istab,VARCHAR,YES,None,None,None


## Vergleich der Verläufe nach Nummern

In [7]:
datum = '2026-05-26'

In [8]:
duck.sql(f"""select distinct linie, kurs::int
         from ivu.rt 
         where datum = '{datum}'
         and linie in (6352,1270)
         order by linie, kurs
         """).df()

,linie,CAST(kurs AS INTEGER)
0,1270,1270008
1,1270,1270013
2,1270,1270014
3,1270,1270019
4,1270,1270020
...,...,...
107,6352,6352515
108,6352,6352902
109,6352,6352904
110,6352,6352924


In [9]:
df = duck.sql(f"""select kurs::int as kurs, nr, hpkt, haltestelle_name
         from ivu.rt 
         where 
         datum = '{datum}'
         -- and linie in (1270)
         -- and kurs in (1226005, 1102202, 1102308)
         and kurs in (6352017, 1270013,6352002)
         and nr < 18

         -- and hpkt in (6002502)
         order by  kurs, nr
         """).df()
df
#df.to_excel('reports/mastzuordnung_lappan.xlsx', index=False)

,kurs,nr,hpkt,haltestelle_name
0,1270013,1,9475005,"Oldenburg, ZOB E"
1,1270013,2,6036201,"Oldenburg, HBF-SÃ¼d A"
2,1270013,3,6002502,"Oldenburg, Lappan B"
3,1270013,4,9481001,"Oldenburg, StaustraÃe A"
4,1270013,5,9492401,"Oldenburg, Schlossplatz A"
5,1270013,6,9558001,"Oldenburg, Am Festungsgraben"
6,1270013,7,9508101,"Oldenburg, Staatsarchiv/Museum"
7,1270013,8,6027801,"Oldenburg, P+R Westfalendamm"
8,1270013,9,9509701,"Munderloh, Stolle"
9,1270013,10,9509801,"Munderloh, BrÃ¼ers"


In [10]:
duck.sql("""
         select hpkt, count(*) anz from (
         select distinct hpkt, haltestelle_name
         from ivu.rt where datum > '2026-05-01' 
         group by all
         order by hpkt)
         group by hpkt
         order by anz desc
         """).df()

,hpkt,anz
0,9453502,3
1,9771601,3
2,9453902,3
3,9771801,3
4,9771501,3
...,...,...
7628,9866602,1
7629,9876901,1
7630,9993801,1
7631,11690302,1


In [11]:
duck.sql("""
         select hpkt, count(*) anz from (
         select distinct hpkt, haltestelle_name
         from ivu.rt where datum > '2026-05-01' 
         group by all
         order by hpkt)
         group by hpkt
         order by anz desc
         """).df()

,hpkt,anz
0,9771007,3
1,9456901,3
2,9771202,3
3,9771802,3
4,9775201,3
...,...,...
7628,9888102,1
7629,10500902,1
7630,10519901,1
7631,11007801,1


In [12]:
duck.sql("""select distinct hpkt, string_agg(distinct haltestelle_name, '#' order by haltestelle_name) as haltestellen, 
         string_agg(distinct linie, '#' order by linie) as linien
         from ivu.rt 
         where datum > '2026-05-01' 
         -- and hpkt = 1750902
         group by all
         order by hpkt """).df().to_excel('reports/haltestellen_und_linien.xlsx', index=False)

In [13]:
duck.sql(f"""create or replace table fahrten as 
         select * 
         from read_parquet('/home/zvbn/python/rt2/out/parquet/prod/fahrten*.parquet')
         where datum >= '{datum}';
         """)

In [40]:
duck.sql("""select lineid, min(datum) as min_date , max(datum) as max_date, count(*) as anz 
         from read_parquet('/home/zvbn/python/rt2/out/parquet/prod/fahrten_*.parquet')
         where lineid like '%220%'
         group by all
         order by max_date asc
         """).df()

,lineid,min_date,max_date,anz
0,de:VBN:220:,2024-08-29,2026-06-01,7043


In [14]:
duck.sql(f"""create or replace table verlauf as 
         select * 
         from read_parquet('/home/zvbn/python/rt2/out/parquet/prod/verlauf_2025_01*.parquet')
         where operday = '2025-01-08'
         and lineshortname in ('330');
         """)

In [15]:
duck.sql("select distinct deviceid from verlauf")

┌───────────────────────────────┐
│           deviceid            │
│            varchar            │
├───────────────────────────────┤
│ 0108-1330007#!ADD!#IVU-Regio# │
│ 0108-1330013#!ADD!#IVU-Regio# │
│ 0108-4398#!ADD!#vwg#          │
│ 0108-1330015#!ADD!#IVU-Regio# │
│ 0108-1330046#!ADD!#IVU-Regio# │
│ 0108-1330054#!ADD!#IVU-Regio# │
│ 0108-1330039#!ADD!#IVU-Regio# │
│ 0108-3702#!ADD!#vwg#          │
│ 0108-1330020#!ADD!#IVU-Regio# │
│ 0108-4918#!ADD!#vwg#          │
│          ·                    │
│          ·                    │
│          ·                    │
│ 0108-8762#!ADD!#vwg#          │
│ 0108-1267#!ADD!#vwg#          │
│ 0108-1330022#!ADD!#IVU-Regio# │
│ 0108-5322#!ADD!#vwg#          │
│ 0108-1330060#!ADD!#IVU-Regio# │
│ 0108-376#!ADD!#vwg#           │
│ 0108-4021#!ADD!#vwg#          │
│ 0108-1330071#!ADD!#IVU-Regio# │
│ 0108-1330078#!ADD!#IVU-Regio# │
│ 0108-1330041#!ADD!#IVU-Regio# │
├───────────────────────────────┤
│      75 rows (20 shown)       │
└─────────────

In [16]:
duck.sql("select * from verlauf").df().to_excel("/home/zvbn/python/rt2/reports/verlauf_2025_01_08.xlsx", index=False)

In [17]:
duck.sql("describe db_dm.basis.linien").df()

,column_name,column_type,null,key,default,extra
0,nummer,VARCHAR,NO,None,None,None
1,von,VARCHAR,NO,None,None,None
2,nach,VARCHAR,YES,None,None,None
3,ueber,VARCHAR,YES,None,None,None
4,genehmigt_nach_paragraph,VARCHAR,YES,None,None,None
5,genehmigung_start,DATE,YES,None,None,None
6,genehmigung_ende,DATE,YES,None,None,None
7,buendel,VARCHAR,YES,None,None,None
8,produkt,VARCHAR,YES,None,None,None
9,anmerkung,VARCHAR,YES,None,None,None


In [18]:
duck.sql("""create or replace table linien as 
         select nummer, buendel, dlid, rbl_li_nr from  db_dm.basis.linien""")

In [19]:
duck.sql("select distinct lineid_short from fahrten order by lineid_short").df()

,lineid_short
0,de:VBN-VGC:910
1,de:VBN:1
2,de:VBN:10
3,de:VBN:101
4,de:VBN:102
...,...
671,de:VBN:S35
672,de:hvv:RB33
673,de:hvv:RB41
674,de:hvv:RE4


In [20]:
duck.sql("show tables from ivu.main;").df()

,name
0,halt_1
1,kal
2,linien_dm
3,rt
4,rt_pre_test_kenn
5,rt_red


In [21]:
duck.sql("describe ivu.main.rt;").df()

,column_name,column_type,null,key,default,extra
0,datum,TIMESTAMP_NS,YES,None,None,None
1,linie,VARCHAR,YES,None,None,None
2,fahrzeug,BIGINT,YES,None,None,None
3,fahrt,DOUBLE,YES,None,None,None
4,kurs,DOUBLE,YES,None,None,None
5,nr,BIGINT,YES,None,None,None
6,hpkt,BIGINT,YES,None,None,None
7,sollabfahrt,VARCHAR,YES,None,None,None
8,sollab,VARCHAR,YES,None,None,None
9,istab,VARCHAR,YES,None,None,None


In [22]:
duck.sql(f"""create or replace table fahrten_ivu as 
         select i.datum,i.linie,i.kurs,i.polizeiliches_kennzeichen, l.*
         from ivu.main.rt i
         left join linien l on i.linie = l.rbl_li_nr 
            where i.datum >= '{datum}'

            group by all
         ---limit 10;
         """)

In [23]:
duck.sql("from fahrten_ivu")

┌─────────────────────┬─────────┬───────────┬───────────────────────────┬─────────┬─────────────┬────────────┬──────────────┐
│        datum        │  linie  │   kurs    │ polizeiliches_kennzeichen │ nummer  │   buendel   │    dlid    │  rbl_li_nr   │
│    timestamp_ns     │ varchar │  double   │          varchar          │ varchar │   varchar   │  varchar   │ decimal(6,0) │
├─────────────────────┼─────────┼───────────┼───────────────────────────┼─────────┼─────────────┼────────────┼──────────────┤
│ 2026-05-29 00:00:00 │ 6403    │ 6403046.0 │ NULL                      │ 403     │ WM Nord     │ de:VBN:403 │         6403 │
│ 2026-05-29 00:00:00 │ 1103    │ 1103001.0 │ DH-WJ-505                 │ 103     │ DH Nordwest │ de:VBN:103 │         1103 │
│ 2026-05-29 00:00:00 │ 1113    │ 1113011.0 │ NULL                      │ 113     │ DH Nordwest │ de:VBN:113 │         1113 │
│ 2026-05-29 00:00:00 │ 1113    │ 1113007.0 │ NULL                      │ 113     │ DH Nordwest │ de:VBN:113 │        

In [ ]:
duck.sql("""select distinct linie 
         from fahrten_ivu
         where linie::text like '_3__'
         order by linie""").df()

## Abfrage einzelner Kennzeichen

In [ ]:
duck.sql("""select datum, kurs, polizeiliches_kennzeichen
         from ivu.main.rt 
         where polizeiliches_kennzeichen like 'DH-R%3324%'
         order by datum desc
         --- limit 10
         """).df()

## Abfrage des Fahrtverlaufs aus IVU.control

In [ ]:
duck.sql("""select datum, linie, kurs::int, polizeiliches_kennzeichen, nr, sollab, istab
         from ivu.main.rt 
         where datum = '2026-05-26' 
         and kurs in (1340113, 1340015)
         order by datum, kurs, nr
         --- limit 10
         """).df()

In [ ]:
duck.sql("select distinct buendel from fahrten_ivu").df()

In [ ]:
duck.sql("""select * 
         from fahrten f
         left join fahrten_ivu i
         on 
            f.datum = i.datum and 
            f.fnr::text = i.kurs::int::text and
            f.lineid_short = i.dlid
         
         where i.buendel = 'AM Ost'
         and f.hasrealtime = false
         order by f.datum, f.fnr
         """).df().to_excel('reports/ohne_echtzeit_amm_ost.xlsx', index=False)

In [ ]:
duck.close()